## Timeboost Markout Repro Notebook

This notebook reproduces the price tables and the final per-auction markout CSV end-to-end.

- Inputs expected under the project directory:
  - `binance_klines/*.csv` (1s OHLC; used to build consolidated prices)
  - `onchain_data/auction_resolved.tsv`
  - `onchain_data/timeboosted_swaps_*.csv`
- Output:
  - `price_tables/prices_consolidated.tsv`
  - `auction_resolved_with_markouts.csv`

Run all cells top-to-bottom.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import polars as pl

ROOT_DIR = Path(__file__).parent
BINANCE_DIR = ROOT_DIR / "binance_klines"
ONCHAIN_DIR = ROOT_DIR / "onchain_data"
PRICE_DIR = ROOT_DIR / "price_tables"
OUTPUT_CSV = ROOT_DIR / "auction_resolved_with_markouts.csv"

# Configure date range (UTC)
START_DATETIME = "2025-06-01 00:00:00"
END_DATETIME = "2025-08-31 23:59:59"
START_TS = int(
    datetime.strptime(START_DATETIME, "%Y-%m-%d %H:%M:%S")
    .replace(tzinfo=timezone.utc)
    .timestamp()
)
END_TS = int(
    datetime.strptime(END_DATETIME, "%Y-%m-%d %H:%M:%S")
    .replace(tzinfo=timezone.utc)
    .timestamp()
)

# Constants
MARKOUT_SECONDS = 5
WEI_PER_ETH = 10**18

# Token addresses
ETH_ADDRESS = "0x82aF49447D8a07e3bd95BD0d56f35241523fBab1"
BTC_ADDRESS = "0x2f2a2543B76A4166549F7aaB2e75Bef0aefC5B0f"
USDC_ADDRESS = "0xaf88d065e77c8cC2239327C5EDb3A432268e5831"
USDCe_ADDRESS = "0xFF970A61A04b1cA14834A43f5dE4533eBDDB5CC8"
USDT_ADDRESS = "0xFd086bC7CD5C481DCC9C85ebE478A1C0b69FCbb9"
SUPPORTED_ADDRS_LOWER = {
    ETH_ADDRESS.lower(),
    BTC_ADDRESS.lower(),
    USDC_ADDRESS.lower(),
    USDCe_ADDRESS.lower(),
    USDT_ADDRESS.lower(),
}

In [ ]:
def detect_timestamp_divisor(example_ts: int) -> int:
    if example_ts >= 1_000_000_000_000_000:
        return 1_000_000
    if example_ts >= 1_000_000_000_000:
        return 1_000
    return 1


def process_binance_file(csv_path: Path, address: str) -> pl.DataFrame | None:
    try:
        head_df = pl.read_csv(
            csv_path,
            has_header=False,
            columns=[0],
            n_rows=1,
            infer_schema_length=1,
            ignore_errors=True,
        )
        if head_df.height == 0:
            return None
        ts_raw = int(head_df.to_series()[0])
        divisor = detect_timestamp_divisor(ts_raw)

        df = pl.read_csv(
            csv_path,
            has_header=False,
            columns=[0, 1],
            new_columns=["timestamp_raw", "open_price"],
            ignore_errors=False,
        )
        out = df.select(
            [
                (pl.col("timestamp_raw").cast(pl.Int64) // divisor).alias("timestamp"),
                pl.col("open_price").cast(pl.Float64).round(8).alias("price"),
            ]
        ).with_columns(pl.lit(address).alias("address"))
        return out
    except Exception:
        return None


def address_for_pair(pair: str) -> str | None:
    upper = pair.upper()
    if upper.startswith("ETH"):
        return ETH_ADDRESS
    if upper.startswith("BTC"):
        return BTC_ADDRESS
    if upper.startswith("USDC"):
        return USDC_ADDRESS
    return None


def build_consolidated_prices() -> Path:
    PRICE_DIR.mkdir(parents=True, exist_ok=True)
    out_path = PRICE_DIR / "prices_consolidated.tsv"
    if out_path.exists():
        out_path.unlink()
    pl.DataFrame(
        {
            "timestamp": pl.Series([], dtype=pl.Int64),
            "price": pl.Series([], dtype=pl.Float64),
            "address": pl.Series([], dtype=pl.String),
        }
    ).write_csv(out_path, separator="\t", include_header=True)
    for entry in sorted(BINANCE_DIR.glob("*.csv")):
        pair = entry.name.split("-")[0]
        address = address_for_pair(pair)
        if address is None:
            continue
        df = process_binance_file(entry, address)
        if df is None or df.height == 0:
            continue
        with out_path.open("a") as f:
            df.write_csv(f, separator="\t", include_header=False)
    return out_path

In [ ]:
def read_swaps_all() -> pl.DataFrame:
    pattern = str(ONCHAIN_DIR / "timeboosted_swaps_*.csv")
    lf = pl.scan_csv(pattern, infer_schema_length=0)
    lf = lf.with_columns(
        [
            pl.col("timestamp").cast(pl.Int64),
            pl.col("tx_hash").cast(pl.String),
            pl.col("token_in").str.to_lowercase().alias("token_in_addr"),
            pl.col("token_out").str.to_lowercase().alias("token_out_addr"),
            pl.col("gas_used").cast(pl.Int64),
            pl.col("effective_gas_price").cast(pl.Int64),
            pl.col("amount_in").cast(pl.Float64),
            pl.col("amount_out").cast(pl.Float64),
        ]
    ).filter(
        pl.col("token_in_addr").is_in(list(SUPPORTED_ADDRS_LOWER))
        | pl.col("token_out_addr").is_in(list(SUPPORTED_ADDRS_LOWER))
    )
    return lf.collect()


def read_prices_supported() -> pl.DataFrame:
    prices = pl.read_csv(
        PRICE_DIR / "prices_consolidated.tsv",
        separator="\t",
        infer_schema_length=0,
        try_parse_dates=False,
    )
    prices = prices.with_columns(
        [
            pl.col("timestamp").cast(pl.Int64),
            pl.col("address").str.to_lowercase().alias("address"),
            pl.col("price").cast(pl.Float64),
        ]
    ).filter(pl.col("address").is_in(list(SUPPORTED_ADDRS_LOWER)))
    # USDC.e mirrors USDC; USDT=1.0 on union of timestamps
    usdc_prices = prices.filter(pl.col("address") == USDC_ADDRESS.lower())
    usdce_prices = usdc_prices.with_columns(
        pl.lit(USDCe_ADDRESS.lower()).alias("address")
    ).select(["timestamp", "price", "address"])
    all_ts = prices.select(pl.col("timestamp")).unique()
    usdt_prices = all_ts.with_columns(
        [
            pl.lit(USDT_ADDRESS.lower()).alias("address"),
            pl.lit(1.0).alias("price"),
        ]
    ).select(["timestamp", "price", "address"])
    prices = prices.select(["timestamp", "price", "address"])
    return pl.concat([prices, usdce_prices, usdt_prices])

In [ ]:
def compute_per_swap_markout(swaps: pl.DataFrame, prices: pl.DataFrame) -> pl.DataFrame:
    p_tp_out = prices.rename({"price": "price_out_tp"})
    swaps_plus = swaps.with_columns(
        (pl.col("timestamp") + MARKOUT_SECONDS).alias("timestamp_plus")
    )
    swaps_out = swaps_plus.join(
        p_tp_out,
        left_on=["token_out_addr", "timestamp_plus"],
        right_on=["address", "timestamp"],
        how="left",
    )
    p_tp_in = prices.rename({"price": "price_in_tp"})
    swaps_io = swaps_out.join(
        p_tp_in,
        left_on=["token_in_addr", "timestamp_plus"],
        right_on=["address", "timestamp"],
        how="left",
    )
    return swaps_io.with_columns(
        (
            pl.col("amount_out") * pl.col("price_out_tp").fill_null(0.0)
            - pl.col("amount_in") * pl.col("price_in_tp").fill_null(0.0)
        ).alias("swap_markout_usd")
    )


def compute_per_tx_minus_gas(
    swaps_with_markout: pl.DataFrame, prices: pl.DataFrame
) -> pl.DataFrame:
    per_tx = swaps_with_markout.group_by("tx_hash").agg(
        [
            pl.col("timestamp").min().alias("tx_timestamp"),
            pl.col("swap_markout_usd").sum().alias("tx_markout_usd"),
            pl.col("gas_used").first().alias("gas_used_first"),
            pl.col("effective_gas_price").first().alias("effective_gas_price_first"),
        ]
    )
    per_tx = per_tx.with_columns(
        [
            (
                pl.col("gas_used_first")
                * pl.col("effective_gas_price_first")
                / WEI_PER_ETH
            ).alias("gas_cost_eth"),
            (pl.col("tx_timestamp") + MARKOUT_SECONDS).alias("tx_timestamp_plus"),
        ]
    )
    eth_prices = (
        prices.filter(pl.col("address") == ETH_ADDRESS.lower())
        .select(["timestamp", "price"])
        .rename({"price": "eth_price_tp"})
    )
    per_tx = per_tx.join(
        eth_prices, left_on="tx_timestamp_plus", right_on="timestamp", how="left"
    )
    return per_tx.with_columns(
        [
            (pl.col("gas_cost_eth") * pl.col("eth_price_tp").fill_null(0.0)).alias(
                "gas_cost_usd"
            ),
            (
                pl.col("tx_markout_usd")
                - pl.col("gas_cost_eth") * pl.col("eth_price_tp").fill_null(0.0)
            ).alias("tx_markout_minus_gas_usd"),
        ]
    )


def read_auction_resolved_filtered() -> pl.DataFrame:
    ar = pl.read_csv(
        ONCHAIN_DIR / "auction_resolved.tsv",
        separator="\t",
        infer_schema_length=0,
        try_parse_dates=False,
    )
    ar = ar.with_columns(
        [
            pl.col("round_start_timestamp").cast(pl.Int64),
            pl.col("round_end_timestamp").cast(pl.Int64),
        ]
    )
    ar = ar.filter(
        (pl.col("round_end_timestamp") >= pl.lit(START_TS))
        & (pl.col("round_end_timestamp") <= pl.lit(END_TS))
    )
    return ar.with_row_index(name="__row_id__")


def map_transactions_to_rounds(per_tx: pl.DataFrame, ar: pl.DataFrame) -> pl.DataFrame:
    per_tx_sorted = per_tx.sort("tx_timestamp")
    ar_sorted = ar.sort("round_start_timestamp")
    joined = per_tx_sorted.join_asof(
        ar_sorted,
        left_on="tx_timestamp",
        right_on="round_start_timestamp",
        strategy="backward",
    )
    mapped = joined.filter(pl.col("tx_timestamp") <= pl.col("round_end_timestamp"))
    return mapped.select(["__row_id__", "tx_markout_minus_gas_usd"])

In [ ]:
# 1) Build consolidated prices from Binance klines
PRICE_DIR.mkdir(parents=True, exist_ok=True)
prices_path = build_consolidated_prices()
prices = read_prices_supported()
print(prices.shape)
prices.head()

In [ ]:
# 2) Read swaps and compute markouts per tx
swaps = read_swaps_all()
swaps_markout = compute_per_swap_markout(swaps, prices)
per_tx = compute_per_tx_minus_gas(swaps_markout, prices)
per_tx.shape

In [ ]:
# 3) Map to rounds, aggregate, add USD columns, and write CSV
ar = read_auction_resolved_filtered()
mapped = map_transactions_to_rounds(per_tx, ar)
summed = mapped.group_by("__row_id__").agg(
    pl.col("tx_markout_minus_gas_usd").sum().alias("agg_markout")
)

out_full = (
    ar.join(summed, on="__row_id__", how="left")
    .with_columns(pl.col("agg_markout").fill_null(0.0))
    .drop("__row_id__")
)

eth_at_start = (
    prices.filter(pl.col("address") == ETH_ADDRESS.lower())
    .select(["timestamp", "price"])
    .rename({"price": "eth_price_at_round_start"})
)

out_full = out_full.join(
    eth_at_start,
    left_on="round_start_timestamp",
    right_on="timestamp",
    how="left",
)

out_full = out_full.with_columns(
    [
        (
            pl.col("first_price_amount").cast(pl.Float64)
            / WEI_PER_ETH
            * pl.col("eth_price_at_round_start").fill_null(0.0)
        ).alias("bid_in_usd"),
        (
            pl.col("price").cast(pl.Float64)
            / WEI_PER_ETH
            * pl.col("eth_price_at_round_start").fill_null(0.0)
        ).alias("payment_in_usd"),
    ]
)

out = out_full.select(
    [
        "round",
        "first_price_bidder",
        "first_price_express_lane_controller",
        "first_price_amount",
        "price",
        "round_start_timestamp",
        "round_end_timestamp",
        "agg_markout",
        "bid_in_usd",
        "payment_in_usd",
    ]
)

out.write_csv(OUTPUT_CSV)
OUTPUT_CSV